# 02. Embedding + DSC + Probe (Image × Regression, ADR-019 재설계)

config마다: pollute(train) → **frozen ResNet18 임베딩 1회** → DSC(임베딩 재사용) + **probe**(train emb→clean test emb).
full finetune 제거 → GPU는 forward-only 임베딩만. raw npz 저장 폐지(RAM·Drive 절감).

In [1]:
# 0-1. Drive 마운트 + GPU
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, gc
import numpy as np, pandas as pd, torch
BASE = '/content/drive/MyDrive/capstone/dsc'
RESULTS_DIR = f'{BASE}/results'
DATA_DIR = f'{BASE}/data/image_regression'
os.makedirs(RESULTS_DIR, exist_ok=True); os.makedirs(DATA_DIR, exist_ok=True)
if BASE not in sys.path: sys.path.insert(0, BASE)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {device} | torch {torch.__version__}')

Mounted at /content/drive
device: cpu | torch 2.11.0+cpu


In [2]:
%pip install -q datasets timm imagehash opencv-python-headless

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 15.2 MB/s eta 0:00:00


In [3]:
# 사전등록 메타 (ADR-018/019)
DATASETS = {
    'UTKFace':      {'hf': 'nu-delta/utkface', 'target': 'age',          'image_col': 'image', 'role': 'tune'},
    'SCUT_FBP5500': {'hf': 'MnLgt/scut-fbp5500',              'target': 'beauty_score', 'image_col': 'image', 'role': 'held-out'},
}
TUNE_DS, HELD_DS = 'UTKFace', 'SCUT_FBP5500'
POLLUTION_LEVELS = [0.1, 0.3, 0.5, 0.7, 0.9]
SAMPLE_CAP = 1000
TEST_CAP = 1000
RANDOM_SEED = 42; ML_SPLIT_SEED = 1; ML_TEST_SIZE = 0.2
print('datasets:', list(DATASETS.keys()), '| levels:', POLLUTION_LEVELS)

datasets: ['UTKFace', 'SCUT_FBP5500'] | levels: [0.1, 0.3, 0.5, 0.7, 0.9]


In [4]:
from datasets import load_dataset
from sklearn.model_selection import train_test_split
def load_hf_split(ds_name):
    meta = DATASETS[ds_name]
    ds = load_dataset(meta['hf'], split='train')
    tr_idx, te_idx = train_test_split(np.arange(len(ds)), test_size=ML_TEST_SIZE, random_state=ML_SPLIT_SEED)
    return ds, meta, tr_idx, te_idx
def to_arrays(ds, meta, indices, sample_cap=None, random_state=1):
    indices = np.asarray(indices)
    if sample_cap and len(indices) > sample_cap:
        rng = np.random.RandomState(random_state)
        indices = indices[rng.permutation(len(indices))[:sample_cap]]
    images, targets = [], []
    for i in indices:
        ex = ds[int(i)]; img = ex[meta['image_col']]
        if hasattr(img, 'convert'): img = img.convert('RGB')
        images.append(np.array(img, dtype=np.uint8)); targets.append(float(ex[meta['target']]))
    return images, targets

In [5]:
# import: DSC + 임베딩추출 + probe + polluters (drive stale 복구)
import importlib
if not os.path.isdir(f'{BASE}/dsc_framework'):
    from google.colab import drive; drive.mount('/content/drive', force_remount=True)
if BASE not in sys.path: sys.path.insert(0, BASE)
importlib.invalidate_caches()
for _m in list(sys.modules):
    if _m.startswith('dsc_framework'): del sys.modules[_m]
if not hasattr(pd.DataFrame, 'append'):
    pd.DataFrame.append = lambda s, o, ignore_index=False, **k: pd.concat([s, o], ignore_index=ignore_index)
from dsc_framework import compute_dsc_image_regression
from dsc_framework.image_cell import _extract_features
from dsc_framework.perf_probe import evaluate_probes
from dsc_framework.image_polluters import (
    CompletenessImagePolluter, NoiseInjectionPolluter, BlurPolluter,
    TargetDistributionSkewImagePolluter, TargetNoiseImagePolluter)
def create_polluters(level, seed=RANDOM_SEED):
    return [('completeness_image', CompletenessImagePolluter(level=level, random_seed=seed)),
            ('noise_injection', NoiseInjectionPolluter(level=level, random_seed=seed)),
            ('blur', BlurPolluter(level=level, random_seed=seed)),
            ('target_distribution_skew', TargetDistributionSkewImagePolluter(level=level, random_seed=seed)),
            ('target_noise', TargetNoiseImagePolluter(level=level, random_seed=seed))]
print('import 완료 (DSC + 임베딩 + probe + polluter 5종)')

Mounted at /content/drive
import 완료 (DSC + 임베딩 + probe + polluter 5종)


In [ ]:
# 메인 루프: 임베딩 1회(DSC+probe 공유, precomputed) + config마다 증분 저장
from time import time
DSC_PATH = f'{RESULTS_DIR}/dsc_scores_image_regression.csv'
PERF_PATH = f'{RESULTS_DIR}/model_performance_image_regression.csv'
dsc_rows, perf_rows = [], []
def save_partial():
    pd.DataFrame(dsc_rows).to_csv(DSC_PATH, index=False)
    pd.DataFrame(perf_rows).to_csv(PERF_PATH, index=False)
t_all = time()
for ds_name in DATASETS:
    print(f'\n=== {ds_name} ===')
    ds, meta, tr_idx, te_idx = load_hf_split(ds_name)
    tr_img, tr_tgt = to_arrays(ds, meta, tr_idx, sample_cap=SAMPLE_CAP, random_state=1)
    te_img, te_tgt = to_arrays(ds, meta, te_idx, sample_cap=TEST_CAP, random_state=1)
    del ds; gc.collect()
    feats_te, idx_te = _extract_features(te_img, sample_cap=TEST_CAP, random_state=1)
    y_te = np.asarray(te_tgt, dtype=float)[idx_te]
    print(f'  train {len(tr_img)} / test {len(te_img)} (test emb {feats_te.shape})')

    def run(pname, level, imgs, tgts):
        feats_tr, idx_tr = _extract_features(imgs, sample_cap=SAMPLE_CAP, random_state=1)
        res = compute_dsc_image_regression(imgs, tgts, sample_cap=SAMPLE_CAP,
                                           precomputed_feats=(feats_tr, idx_tr))
        res.pop('metrics', None)
        dsc_rows.append({'dataset': ds_name, 'polluter': pname, 'level': level, **res})
        y_tr = np.asarray(tgts, dtype=float)[idx_tr]
        scores = evaluate_probes(feats_tr, y_tr, feats_te, y_te, 'regression')
        for m, sc in scores.items():
            if m.startswith('_'): continue
            perf_rows.append({'dataset': ds_name, 'polluter': pname, 'level': level,
                              'method': 'probe', 'model': m, 'score': sc})
        del feats_tr; gc.collect()
        return res['score'], scores

    s, sc = run('none', 0.0, tr_img, tr_tgt)
    print(f'  baseline DSC={s:.2f} probe={ {k:v for k,v in sc.items() if not k.startswith("_")} }')
    save_partial()
    for level in POLLUTION_LEVELS:
        for pname, pol in create_polluters(level):
            t0 = time()
            try:
                pi, pt = pol.pollute(tr_img, tr_tgt)
                s, _ = run(pname, level, pi, pt)
                print(f'  {pname:26s} L{level:.1f} DSC={s:6.2f} ({time()-t0:.0f}s)')
                del pi, pt; gc.collect()
                save_partial()
            except Exception as e:
                print(f'  {pname:26s} L{level:.1f} ERROR: {e}')
    del tr_img, tr_tgt, te_img, te_tgt, feats_te; gc.collect()
    print(f'  [{ds_name} done: DSC {len(dsc_rows)}, probe {len(perf_rows)}]')
save_partial()
print(f'\n완료 ({time()-t_all:.0f}s): DSC {len(dsc_rows)}, probe perf {len(perf_rows)}')


=== UTKFace ===


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/467 [00:00<?, ?B/s]

data/train-00000-of-00003.parquet:   0%|          | 0.00/343M [00:00<?, ?B/s]

data/train-00001-of-00003.parquet:   0%|          | 0.00/343M [00:00<?, ?B/s]

...

  completeness_image         L0.1 DSC= 82.96 (651s)
  noise_injection            L0.1 DSC= 85.66 (659s)


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(


  blur                       L0.1 DSC= 82.69 (626s)
  target_distribution_skew   L0.1 DSC= 82.98 (602s)
  target_noise               L0.1 DSC= 83.11 (630s)


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(


  completeness_image         L0.3 DSC= 82.59 (636s)
  noise_injection            L0.3 DSC= 84.56 (653s)


In [ ]:
# (증분 저장이 루프 안에서 수행됨) 최종 확인
print('DSC:', len(pd.read_csv(DSC_PATH)), '행 | perf:', len(pd.read_csv(PERF_PATH)), '행')
print('--- 02 완료 → 03(spot-check, GPU) 또는 04(scoreboard, CPU) ---')